In [8]:
# Cell 1: Import the tools we need
import pandas as pd
import json
import glob
import os

In [9]:
# Cell 2: Load all 7 files and merge them into one table
all_files = glob.glob("/content/Streaming_History_Audio_*.json")

dfs = []
for file in all_files:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)
    dfs.append(pd.DataFrame(data))

df = pd.concat(dfs, ignore_index=True)

print(f"Files found: {len(all_files)}")
print(f"Total records: {len(df)}")

Files found: 7
Total records: 99057


In [10]:
# Cell 3: Preview the data
print("Shape:", df.shape)
print()
print("Columns:", df.columns.tolist())
print()
df.head(3)

Shape: (99057, 23)

Columns: ['ts', 'platform', 'ms_played', 'conn_country', 'ip_addr', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name', 'spotify_track_uri', 'episode_name', 'episode_show_name', 'spotify_episode_uri', 'audiobook_title', 'audiobook_uri', 'audiobook_chapter_uri', 'audiobook_chapter_title', 'reason_start', 'reason_end', 'shuffle', 'skipped', 'offline', 'offline_timestamp', 'incognito_mode']



,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,episode_name,...,audiobook_uri,audiobook_chapter_uri,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode
0,2024-02-12T08:05:11Z,android,163604,IN,152.58.3.247,Tired Eyes,Kado,Tired Eyes,spotify:track:0MSLJOWljfQr067PYyndK9,None,...,None,None,None,trackdone,trackdone,True,False,False,1.707725e+09,False
1,2024-02-12T08:06:08Z,android,65600,IN,152.58.3.247,crossed out,Yung Crusha,crossed out,spotify:track:5JnDp6C0UTC3X1cRH2RNsY,None,...,None,None,None,trackdone,trackdone,True,False,False,1.707725e+09,False
2,2024-02-12T08:08:14Z,android,135616,IN,152.58.3.247,50/50,Jude Barclay,another year,spotify:track:1Wmedr7gGmDYkDiv8kJODa,None,...,None,None,None,trackdone,trackdone,True,False,False,1.707725e+09,False


In [11]:
# Cell 4: Drop columns we don't need
cols_to_drop = [
    "episode_name", "episode_show_name", "spotify_episode_uri",
    "audiobook_title", "audiobook_uri",
    "audiobook_chapter_uri", "audiobook_chapter_title",
    "ip_addr", "offline_timestamp", "spotify_track_uri"
]

df.drop(columns=cols_to_drop, inplace=True)

print("Columns remaining:", df.columns.tolist())
print("New shape:", df.shape)

Columns remaining: ['ts', 'platform', 'ms_played', 'conn_country', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name', 'reason_start', 'reason_end', 'shuffle', 'skipped', 'offline', 'incognito_mode']
New shape: (99057, 13)


In [12]:
# Cell 5: Rename columns
df.rename(columns={
    "ts": "timestamp",
    "ms_played": "ms_played",
    "conn_country": "country",
    "master_metadata_track_name": "track_name",
    "master_metadata_album_artist_name": "artist_name",
    "master_metadata_album_album_name": "album_name",
    "incognito_mode": "incognito"
}, inplace=True)

print("Updated columns:", df.columns.tolist())

Updated columns: ['timestamp', 'platform', 'ms_played', 'country', 'track_name', 'artist_name', 'album_name', 'reason_start', 'reason_end', 'shuffle', 'skipped', 'offline', 'incognito']


In [13]:
# Cell 6: Parse timestamps and extract time features
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

df["year"]        = df["timestamp"].dt.year
df["month"]       = df["timestamp"].dt.month
df["month_name"]  = df["timestamp"].dt.strftime("%b")
df["day_of_week"] = df["timestamp"].dt.day_name()
df["hour"]        = df["timestamp"].dt.hour
df["date"]        = df["timestamp"].dt.date

print("Sample timestamps:")
print(df[["timestamp", "year", "month", "month_name", "day_of_week", "hour", "date"]].head(5))

Sample timestamps:
                  timestamp  year  month month_name day_of_week  hour  \
0 2024-02-12 08:05:11+00:00  2024      2        Feb      Monday     8   
1 2024-02-12 08:06:08+00:00  2024      2        Feb      Monday     8   
2 2024-02-12 08:08:14+00:00  2024      2        Feb      Monday     8   
3 2024-02-12 08:10:41+00:00  2024      2        Feb      Monday     8   
4 2024-02-12 08:12:43+00:00  2024      2        Feb      Monday     8   

         date  
0  2024-02-12  
1  2024-02-12  
2  2024-02-12  
3  2024-02-12  
4  2024-02-12  


In [14]:
# Cell 7: Convert ms_played to minutes
df["minutes_played"] = (df["ms_played"] / 1000 / 60).round(2)

print("Sample:")
print(df[["track_name", "ms_played", "minutes_played"]].head(5))

Sample:
           track_name  ms_played  minutes_played
0          Tired Eyes     163604            2.73
1         crossed out      65600            1.09
2               50/50     135616            2.26
3            Problems     155464            2.59
4  life's been boring     131265            2.19


In [15]:
# Cell 8: Feature engineering
# Skip flag
df["is_skip"] = (df["skipped"] == True) & (df["ms_played"] < 30000)

# Session bucket
def session_bucket(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

df["session_bucket"] = df["hour"].apply(session_bucket)

print("Skip flag distribution:")
print(df["is_skip"].value_counts())
print()
print("Session bucket distribution:")
print(df["session_bucket"].value_counts())

Skip flag distribution:
is_skip
False    72947
True     26110
Name: count, dtype: int64

Session bucket distribution:
session_bucket
Morning      40492
Afternoon    34241
Evening      16831
Night         7493
Name: count, dtype: int64


In [16]:
# Cell 9: Filter and clean
# Keep only rows where a track name exists (removes podcast/interrupted rows)
before_filter = len(df)
df = df[df["track_name"].notna()].copy()
after_filter = len(df)

print(f"Rows removed (no track name): {before_filter - after_filter}")

# Remove duplicates
before_dedup = len(df)
df.drop_duplicates(inplace=True)
after_dedup = len(df)

print(f"Duplicate rows removed: {before_dedup - after_dedup}")
print(f"Final record count: {len(df)}")

Rows removed (no track name): 9
Duplicate rows removed: 172
Final record count: 98876


In [17]:
# Cell 10: Export clean CSV
df.to_csv("spotify_cleaned.csv", index=False)

print("File saved: spotify_cleaned.csv")
print(f"Final shape: {df.shape[0]} rows x {df.shape[1]} columns")
print()
print("Final columns:")
print(df.columns.tolist())

File saved: spotify_cleaned.csv
Final shape: 98876 rows x 22 columns

Final columns:
['timestamp', 'platform', 'ms_played', 'country', 'track_name', 'artist_name', 'album_name', 'reason_start', 'reason_end', 'shuffle', 'skipped', 'offline', 'incognito', 'year', 'month', 'month_name', 'day_of_week', 'hour', 'date', 'minutes_played', 'is_skip', 'session_bucket']
